# Notebook 01: Data Exploration and Descriptive Statistics
## Study: NCAL-2026-1259 (Computer Assisted Language Learning)
### Evaluating the Impact of Generative AI on EFL Writing Quality, Self-Regulated Learning (SRL), and AI Literacy

---

### Objectives:
1. Environment setup and data ingestion (`Hypothetical_EFL_AI_Study_Data.csv`).
2. Data sanity checks (missing values, types, distribution ranges, sample sizes per group).
3. Compute Table 3: Pre-test and Post-test Descriptive Statistics (Mean, SD, Min, Max, Skewness, Kurtosis) stratified by Group (Experimental vs Control).
4. Export publication-ready descriptive tables.

In [ ]:
import os
import sys
import numpy as np
import pandas as pd
import scipy.stats as stats

# Configure display options
pd.set_option('display.max_columns', 20)
pd.set_option('display.width', 1000)
pd.set_option('display.float_format', lambda x: f'{x:.3f}')

print("Python version:", sys.version.split()[0])
print("Pandas version:", pd.__version__)
print("NumPy version:", np.__version__)

### Step 1: Locate and Ingest the Dataset

In [ ]:
DATA_CANDIDATES = [
    "data/raw/Hypothetical_EFL_AI_Study_Data.csv",
    "data/Hypothetical_EFL_AI_Study_Data.csv",
    "Hypothetical_EFL_AI_Study_Data.csv",
    "/mnt/data/Hypothetical_EFL_AI_Study_Data.csv",
    "../Hypothetical_EFL_AI_Study_Data.csv"
]

def load_data():
    for p in DATA_CANDIDATES:
        if os.path.exists(p):
            print(f"[INFO] Ingesting dataset from: {p}")
            df = pd.read_csv(p)
            df['Group'] = df['Group'].astype(str).str.strip().str.capitalize()
            return df
    raise FileNotFoundError("Data file not found in candidates: " + str(DATA_CANDIDATES))

df = load_data()
print(f"Dataset shape: {df.shape[0]} rows, {df.shape[1]} columns")
df.head(10)

### Step 2: Data Audit and Validation

In [ ]:
print("--- Missing Values Check ---")
print(df.isnull().sum())
print("
--- Sample Sizes by Group ---")
print(df['Group'].value_counts())
print("
--- Data Types ---")
print(df.dtypes)

### Step 3: Compute Summary Statistics (Table 3)

In [ ]:
variables = [
    ('Writing_Pre', 'Writing Post'),
    ('Writing_Post', 'Writing Post'),
    ('SRL_Pre', 'SRL Pre'),
    ('SRL_Post', 'SRL Post'),
    ('AI_Pre', 'AI Pre'),
    ('AI_Post', 'AI Post')
]

vars_list = ['Writing_Pre', 'Writing_Post', 'SRL_Pre', 'SRL_Post', 'AI_Pre', 'AI_Post']

table3_rows = []
for grp_name, grp_df in df.groupby('Group'):
    for v in vars_list:
        data = grp_df[v].dropna()
        n = len(data)
        mean = data.mean()
        sd = data.std(ddof=1)
        se = sd / np.sqrt(n)
        median = data.median()
        skew = stats.skew(data, bias=False)
        kurt = stats.kurtosis(data, bias=False)
        table3_rows.append({
            'Group': grp_name,
            'Variable': v,
            'N': n,
            'Mean': mean,
            'Std_Dev': sd,
            'Std_Error': se,
            'Median': median,
            'Min': data.min(),
            'Max': data.max(),
            'Skewness': skew,
            'Kurtosis': kurt
        })

table3_df = pd.DataFrame(table3_rows)
table3_df

### Step 4: Export Descriptive Statistics

In [ ]:
os.makedirs("outputs", exist_ok=True)
output_path = "outputs/Table3_Descriptive_Statistics.csv"
table3_df.to_csv(output_path, index=False)
print(f"[SUCCESS] Table 3 exported to: {output_path}")